# 2 — Download Models

Pull the runnable models decided by the hardware check through Ollama.

The **candidate models** are defined in `config.py` (`CANDIDATE_MODELS`) — the single source of truth. `3_1_check_server_capabilities.ipynb` filters them against this server's memory and writes `config/runnable_models.json`. This notebook only **executes that decision**: it pulls each runnable model, verifies the installation, and rewrites the config to keep only the models that were actually pulled — so `run_extraction.py` never attempts a model that is not installed.

**Prerequisites:** run `3_1_check_server_capabilities.ipynb` first, and make sure the Ollama server is running (`ollama serve`).

In [ ]:
import os
import json
import shutil
import subprocess

from clinical_notes_extraction.config import RUNNABLE_MODELS_FILE

## 1 - Install the Ollama

In [ ]:
# 1. Update PATH first so the notebook knows where to check
os.environ["PATH"] = os.path.expanduser("~/.local/bin:") + os.environ["PATH"]

# 2. Check if ollama is already installed in your path
if shutil.which("ollama"):
    print("Ollama is already installed! Skipping download and extraction.")
else:
    print("Ollama not found. Beginning clean installation...")
    
    # Clean up any previous broken downloads first
    !rm -f ~/.local/bin/ollama ~/ollama.tar.zst

    # Download the official Linux archive (.tar.zst)
    print("Downloading archive...")
    !curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst -o ~/ollama.tar.zst

    # Extract the archive into a temporary folder inside your account
    print("Extracting files...")
    !mkdir -p ~/ollama_tmp
    !tar -xf ~/ollama.tar.zst -C ~/ollama_tmp

    # Move the executable binary straight into your existing local bin folder
    !mv ~/ollama_tmp/bin/ollama ~/.local/bin/
    !chmod +x ~/.local/bin/ollama

    # Clean up the temporary installation files
    !rm -rf ~/ollama_tmp ~/ollama.tar.zst
    print("Installation complete!")

# 3. Double-check and enforce that it works with an assert statement
assert shutil.which("ollama") is not None, "Installation verification failed! Ollama is not in PATH."

# 4. Print the current running version
!ollama --version

## 2 - Pre-flight checks

Ollama must be installed, and the runnable-models config must exist.

In [ ]:
ollama_path = shutil.which("ollama")

if ollama_path is None:
    print("Ollama not found on PATH.")
    print("Check in a terminal with: which ollama")
    print("If it works there but not here, Jupyter is using a different PATH.")
else:
    print(f"Ollama found at: {ollama_path}")

    if not RUNNABLE_MODELS_FILE.exists():
        print(f"{RUNNABLE_MODELS_FILE} not found — run notebook 00 first.")
    else:
        with open(RUNNABLE_MODELS_FILE) as f:
            config = json.load(f)
        print(f"Memory budget: ~{config['memory_budget_gb']} GB ({config['budget_source']})")
        print(f"Models to pull: {config['models']}")

## 3 - Pull the models

Each pull streams its progress to the terminal running Jupyter. Failed pulls are excluded from the final list rather than aborting the run.

In [ ]:
import ollama

# 1. Initialize the client matching your custom port
client = ollama.Client(host='http://127.0.0.1:11434')

# 2. Get a set of all model names currently installed locally
try:
    local_models_response = client.list()
    # Extract names (handles stripping tags if needed, though exact matching is safest)
    installed_models = {m['model'] for m in local_models_response.get('models', [])}
except Exception as e:
    print(f"Could not connect to Ollama server to list models: {e}")
    installed_models = set()

pulled = []

for model in config["models"]:
    # Check if the model (or its variant) is already present in your local list
    # Note: Ollama client might append ':latest' if no tag is specified.
    model_tag = model if ":" in model else f"{model}:latest"
    
    if model in installed_models or model_tag in installed_models:
        print(f"\n{model} is already installed — skipping.")
        pulled.append(model)
        continue
        
    print(f"\nPulling {model} ...")
    # Explicitly pass the OLLAMA_HOST environment variable so subprocess knows your custom port
    import os
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "127.0.0.1:11434"
    
    result = subprocess.run(["ollama", "pull", model], env=env)
    if result.returncode == 0:
        pulled.append(model)
    else:
        print(f"WARNING: pull of {model} failed — excluded from the experiment.")

print(f"\nSuccessfully verified/pulled: {pulled}")

In [ ]:
print(f"Installed models: {config['models']}\n")
subprocess.run(["ollama", "list"])